# 03 - Backfill textual do Congresso

Valida o texto integral em marco de 2000 e executa a esteira mensal `CN` ate `2026-07-13` com `run_id` proprio.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
ACTIVE_CONFIG_PATH = DATA_ROOT / "operations" / "atualizacao" / "active.json"
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # Opcional: branch, tag ou commit. Vazio usa o default remoto.

os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
for name in ["raw", "checkpoints", "logs", "manifests", "processed", "operations/atualizacao"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("DATA_ROOT:", DATA_ROOT)
print("Repositorio:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
EXPECTED_CYCLE_ID = "20260713"
if not ACTIVE_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Controle ativo ausente: {ACTIVE_CONFIG_PATH}. Execute o caderno 00 primeiro.")
CONFIG = json.loads(ACTIVE_CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["schema_version"] == 1
assert CONFIG["cycle_id"] == EXPECTED_CYCLE_ID, CONFIG["cycle_id"]
assert CONFIG["window"] == {"data_inicio": "2026-05-01", "data_fim": "2026-07-13"}
assert CONFIG["data_inicio"] == CONFIG["window"]["data_inicio"]
assert CONFIG["data_fim"] == CONFIG["window"]["data_fim"]
assert Path(CONFIG["data_root"]) == DATA_ROOT
RUNS = {item["key"]: item for item in CONFIG["collection_runs"]}
print("Ciclo ativo:", CONFIG["cycle_id"], CONFIG["window"])

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone

TERMINAL_STATUSES = {"completed"}

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

def manifest_for(run):
    return DATA_ROOT / "manifests" / f"{run['run_id']}.json"

def checkpoint_for(run):
    return DATA_ROOT / "checkpoints" / run["source"] / f"{run['dataset']}.json"

def unresolved_partitions(run):
    checkpoint = read_json(checkpoint_for(run)) or {}
    current = (checkpoint.get("runs") or {}).get(run["run_id"], {}) or {}
    failed = set((current.get("failed_partitions") or {}).keys())
    completed = set((current.get("completed_partitions") or {}).keys())
    return sorted(failed - completed)

def assert_collection_complete(run):
    manifest = read_json(manifest_for(run))
    assert manifest is not None, f"Manifest final ausente: {manifest_for(run)}"
    assert manifest.get("run_id") == run["run_id"]
    assert manifest.get("status") in TERMINAL_STATUSES, (run["key"], manifest.get("status"))
    assert manifest.get("mode") == "prod", (run["key"], manifest.get("mode"))
    assert manifest.get("sample") is False, (run["key"], manifest.get("sample"))
    assert manifest.get("data_inicio") == run["data_inicio"], (run["key"], manifest.get("data_inicio"))
    assert manifest.get("data_fim") == run["data_fim"], (run["key"], manifest.get("data_fim"))
    unresolved = unresolved_partitions(run)
    assert not unresolved, f"Particoes falhas nao resolvidas em {run['key']}: {unresolved[:20]}"
    return manifest

def show_run_state(run, tail_lines=5):
    final = read_json(manifest_for(run))
    autosave_path = DATA_ROOT / "manifests" / f"{run['run_id']}.autosave.json"
    autosave = read_json(autosave_path)
    log_path = DATA_ROOT / "logs" / f"{run['run_id']}.jsonl"
    tail = log_path.read_text(encoding="utf-8").splitlines()[-tail_lines:] if log_path.exists() else []
    print(run["key"], {
        "manifest": str(manifest_for(run)),
        "status": final.get("status") if final else None,
        "autosave_status": autosave.get("status") if autosave else None,
        "unresolved": unresolved_partitions(run),
        "log_tail": tail,
    })

def collector_command(run, *extra):
    return [
        sys.executable, "-u", "-m", run["module"],
        "--mode", "prod",
        "--output-dir", str(DATA_ROOT),
        "--data-inicio", run["data_inicio"],
        "--data-fim", run["data_fim"],
        "--run-id", run["run_id"],
        "--no-sample", "--resume", *extra,
    ]

def run_streamed(command, label):
    print(f"\n=== {label} ===", flush=True)
    print(" ".join(map(str, command)), flush=True)
    completed = subprocess.run(list(map(str, command)), check=False)
    returncode = completed.returncode
    print(f"=== retorno {returncode}: {label} ===", flush=True)
    return returncode

@contextmanager
def dataset_lock(run):
    lock_root = DATA_ROOT / "operations" / "atualizacao" / "locks"
    lock_root.mkdir(parents=True, exist_ok=True)
    lock_path = lock_root / f"{run['source']}__{run['dataset']}.json"
    payload = {
        "cycle_id": CONFIG["cycle_id"],
        "run_id": run["run_id"],
        "source": run["source"],
        "dataset": run["dataset"],
        "started_at": datetime.now(timezone.utc).isoformat(),
    }
    try:
        with lock_path.open("x", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
            handle.write("\n")
    except FileExistsError as exc:
        raise RuntimeError(f"Dataset ja bloqueado por outra sessao: {lock_path}\n{lock_path.read_text()}") from exc
    try:
        yield
    finally:
        if lock_path.exists() and read_json(lock_path) == payload:
            lock_path.unlink()

def run_collector(run, *extra):
    with dataset_lock(run):
        return run_streamed(collector_command(run, *extra), run["key"])

def require_explicit_confirmation(enabled, confirmation):
    if enabled:
        assert confirmation == EXPECTED_CYCLE_ID, "Digite o cycle_id na variavel CONFIRMAR_CICLO."

def assert_parlamentares_ready():
    run_id = CONFIG["processing_run_ids"]["parlamentares"]
    manifest_path = DATA_ROOT / "processed" / "manifests" / f"{run_id}-parlamentares.json"
    periodos_path = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
    manifest = read_json(manifest_path)
    assert manifest and manifest.get("run_id") == run_id and manifest.get("dataset_version") == "v1", manifest_path
    assert periodos_path.exists(), periodos_path
    return manifest

In [ ]:
RODAR_VALIDACAO_CURTA = False
CONFIRMAR_VALIDACAO = ""
if RODAR_VALIDACAO_CURTA:
    assert CONFIRMAR_VALIDACAO == EXPECTED_CYCLE_ID
    smoke_root = REPO_DIR / "data" / "dev" / "congresso_textos_20260713"
    command = [
        sys.executable, "-u", "-m", "coleta.senado.congresso_discursos.collect",
        "--mode", "dev", "--output-dir", str(smoke_root),
        "--data-inicio", "2000-03-01", "--data-fim", "2000-03-31",
        "--run-id", "smoke-congresso-textos-20260713", "--sample-limit", "3", "--resume",
    ]
    assert run_streamed(command, "smoke textual do Congresso") == 0
    raw_paths = list((smoke_root / "raw" / "senado" / "congresso_discursos").glob("ano=*/mes=*/*.jsonl"))
    records = [json.loads(line) for path in raw_paths for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert records and all(item["record_type"] == "pronunciamento_texto" for item in records)
    assert all(str(item["payload"].get("texto") or "").strip() for item in records)
    metadata_path = smoke_root / "raw" / "senado" / "congresso_discursos" / "metadata" / "smoke-congresso-textos-20260713.jsonl"
    metadata = [json.loads(line) for line in metadata_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert metadata and all(item["request"]["params"].get("siglaCasa") == "CN" for item in metadata)
    print("Smoke aprovado:", len(records), "textos")

In [ ]:
RODAR_BACKFILL = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(RODAR_BACKFILL, CONFIRMAR_CICLO)
run = RUNS["congresso_textos"]
if RODAR_BACKFILL:
    assert_parlamentares_ready()
    rc = run_collector(run)
    assert rc == 0
    assert_collection_complete(run)
else:
    print("Backfill protegido: RODAR_BACKFILL=False")

## Auditoria do corpus textual

In [ ]:
if manifest_for(run).exists():
    manifest = assert_collection_complete(run)
    show_run_state(run)
    corpus = DATA_ROOT / "raw" / "senado" / "congresso_discursos"
    monthly = list(corpus.glob("ano=*/mes=*/*.jsonl"))
    queue = corpus / "transcription_queue" / f"{run['run_id']}.jsonl"
    print("Manifest:", manifest_for(run))
    print("Arquivos mensais:", len(monthly))
    print("Fila de transcricao:", queue, queue.exists())
    assert monthly, "Manifest final sem arquivos textuais mensais."
else:
    print("Backfill ainda pendente:", manifest_for(run))